<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Recommendation

In [247]:
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import MinMaxScaler

In [248]:
import joblib

ARTIFACT_PATH = "/content/drive/MyDrive/CineAI/Artifact"

movie_data = joblib.load(ARTIFACT_PATH+"/movie_data.pkl")

popular_movies = joblib.load(ARTIFACT_PATH+"/popular_movies.pkl")

svd_model = joblib.load(ARTIFACT_PATH+"/svd_model.pkl")

knn_model = joblib.load(ARTIFACT_PATH+"/knn_content_model.pkl")

tfidf_matrix = joblib.load(ARTIFACT_PATH+"/tfidf_matrix.pkl")

tfidf_vectorizer = joblib.load(ARTIFACT_PATH+"/tfidf_vectorizer.pkl")

title_lookup = joblib.load(ARTIFACT_PATH+"/title_lookup.pkl")

title_to_index = joblib.load(ARTIFACT_PATH+"/title_to_index.pkl")

user_encoder = joblib.load(ARTIFACT_PATH+"/user_encoder.pkl")

movie_encoder = joblib.load(ARTIFACT_PATH+"/movie_encoder.pkl")

user_features = joblib.load(ARTIFACT_PATH + "/user_features.pkl")
movie_features = joblib.load(ARTIFACT_PATH + "/movie_features.pkl")


In [249]:
ratings = pd.read_csv(
    "/content/drive/MyDrive/CineAI/ml-32m/ml-32m/ratings.csv"
)

In [250]:
movie_stats = pd.read_pickle("/content/drive/MyDrive/CineAI/Artifact/movie_stats.pkl")

# Popularity Recommendation

In [251]:
def calculate_imdb_score(df, quantile=0.90):
    """
    Calculates IMDb Weighted Rating.

    Parameters
    ----------
    df : DataFrame
    quantile : float
        Minimum vote percentile.

    Returns
    -------
    DataFrame
    """

    df = df.copy()

    C = df["average_rating"].mean()

    m = df["rating_count"].quantile(quantile)

    df["weighted_rating"] = (
        (
            df["rating_count"] /
            (df["rating_count"] + m)
        ) * df["average_rating"]
    ) + (
        (
            m /
            (df["rating_count"] + m)
        ) * C
    )

    return df

In [252]:
popular_movies = calculate_imdb_score(movie_stats)

In [253]:
popular_movies = (
    popular_movies
    .sort_values(
        by="weighted_rating",
        ascending=False
    )
    .reset_index(drop=True)
)

In [254]:
def recommend_popular(top_n=500):

    columns = [
        "movieId",
        "clean_title",
        "genres",
        "average_rating",
        "rating_count",
        "weighted_rating"
    ]

    return popular_movies[columns].head(top_n)

In [255]:
popular = recommend_popular(20)
popular.head()

,movieId,clean_title,genres,average_rating,rating_count,weighted_rating
0,318,"Shawshank Redemption, The",Crime|Drama,4.404614,102929,4.401238
1,159817,Planet Earth,Documentary,4.444369,2948,4.332311
2,858,"Godfather, The",Crime|Drama,4.317030,66440,4.312134
3,170705,Band of Brothers,Action|Drama|War,4.426539,2811,4.310914
4,202439,Parasite,Comedy|Drama,4.312254,11670,4.284956


# Cold Start

In [256]:
def recommend_cold_start(popular_df, top_n=20):
    """
    Recommend movies for brand-new users.

    Uses popularity only because the user has no history.
    """

    cold_start = (
        popular_df
        .sort_values(
            by=[
                "weighted_rating",
                "rating_count",
                "average_rating"
            ],
            ascending=False
        )
        .head(top_n)
    )

    return cold_start

In [257]:
USER_ID = 1
def is_new_user(user_id, ratings):

    if user_id is None:
        return True

    if user_id not in ratings["userId"].values:
        return True

    return False

def is_sparse_user(
    user_id,
    ratings,
    threshold=10
):

    if is_new_user(user_id, ratings):
        return False

    n = len(
        ratings[
            ratings["userId"] == user_id
        ]
    )

    return n < threshold

In [258]:
USER_ID =1
if is_new_user(USER_ID, ratings):

    print("Cold Start User")

    final_recommendations = recommend_cold_start(
        popular_movies,
        top_n=500
    )

    display(final_recommendations)

    raise SystemExit()

# Collabaorative Recommendation

In [259]:
def rank_candidates(
    candidate_indices,
    scores,
    top_n=500
):

    top_indices = candidate_indices[:top_n]

    movie_ids = movie_encoder.inverse_transform(
        top_indices
    )

    recommendations = (
        movie_data[
            movie_data.movieId.isin(
                movie_ids
            )
        ]
        .copy()
    )

    score_map = {
        movie_encoder.inverse_transform([idx])[0]:
        scores[idx]
        for idx in top_indices
    }

    recommendations["svd_score"] = (
        recommendations.movieId
        .map(score_map)
    )

    return recommendations.sort_values(
        "svd_score",
        ascending=False
    )

In [260]:
def filter_seen_movies(
    user_id,
    candidate_indices,
    scores
):

    watched = ratings.loc[
        ratings.userId == user_id,
        "movieId"
    ]

    watched = watched[
        watched.isin(
            movie_encoder.classes_
        )
    ]

    watched_indices = set(
        movie_encoder.transform(
            watched
        )
    )

    filtered = [
        idx
        for idx in candidate_indices
        if idx not in watched_indices
    ]

    return filtered

In [261]:
def generate_candidates(
    user_id,
    candidate_size=500
):

    if user_id not in user_encoder.classes_:
        raise ValueError("Unknown User ID")

    user_index = user_encoder.transform(
        [user_id]
    )[0]

    user_vector = user_features[user_index]

    scores = movie_features @ user_vector

    top_candidates = np.argsort(
        scores
    )[::-1][:candidate_size]

    return top_candidates, scores

In [262]:
def recommend_for_user_collaborative(
    user_id,
    top_n=500
):

    candidates, scores = generate_candidates(
        user_id
    )

    filtered_candidates = filter_seen_movies(
        user_id,
        candidates,
        scores
    )

    recommendations = rank_candidates(
        filtered_candidates,
        scores,
        top_n
    )

    recommendations = (
        recommendations
        .sort_values("svd_score", ascending=False)
        .reset_index(drop=True)
    )

    recommendations.index.name = "Rank"

    return recommendations

In [263]:
collab = recommend_for_user_collaborative(user_id=1, top_n=20)
collab.head()

,movieId,title,genres,year,clean_title,genres_list,imdbId,tmdbId,tag,combined_features,svd_score
Rank,,,,,,,,,,,
0,858,"Godfather, The (1972)",Crime|Drama,1972.0,"Godfather, The","[Crime, Drama]",68646,238.0,al pacino atmospheric great acting masterpiece...,"Godfather, The Crime Drama Al Pacino atmospher...",2.863783
1,1222,Full Metal Jacket (1987),Drama|War,1987.0,Full Metal Jacket,"[Drama, War]",93058,600.0,vietnam war tumey s dvds imdb top 250 stanley ...,Full Metal Jacket Drama War Vietnam War Tumey'...,2.771993
2,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,1968.0,2001: A Space Odyssey,"[Adventure, Drama, Sci-Fi]",62622,62.0,atmospheric space stanley kubrick ambiguous at...,2001: A Space Odyssey Adventure Drama Sci-Fi a...,2.545833
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,1964.0,Dr. Strangelove or: How I Learned to Stop Worr...,"[Comedy, War]",57012,935.0,black and white stanley kubrick classic dark c...,Dr. Strangelove or: How I Learned to Stop Worr...,2.479119
4,2858,American Beauty (1999),Drama|Romance,1999.0,American Beauty,"[Drama, Romance]",169547,14.0,bittersweet great acting kevin spacey loneline...,American Beauty Drama Romance bittersweet grea...,2.291048


# Content Recommedation

In [264]:
from sklearn.metrics.pairwise import linear_kernel
def recommend_content(title, top_n=500):

    matches = movie_data[
        movie_data["clean_title"]
        .str.lower() == title.lower()
    ]

    if matches.empty:
        print("Movie not found.")
        return None

    idx = matches.index[0]

    cosine_scores = linear_kernel(
        tfidf_matrix[idx:idx + 1],
        tfidf_matrix
    ).flatten()

    similar_indices = np.argsort(cosine_scores)[::-1]

    similar_indices = similar_indices[1:top_n + 1]

    recommendations = movie_data.iloc[
        similar_indices
    ][["movieId", "clean_title", "genres", "year"]].copy()

    recommendations["similarity_score"] = (
        cosine_scores[similar_indices]
    )

    return recommendations

In [265]:
content = recommend_content("Toy Story", 20)
content.head()

,movieId,clean_title,genres,year,similarity_score
3021,3114,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999.0,0.892434
2264,2355,"Bug's Life, A",Adventure|Animation|Children|Comedy,1998.0,0.806176
14815,78499,Toy Story 3,Adventure|Animation|Children|Comedy|Fantasy|IMAX,2010.0,0.711669
39850,157296,Finding Dory,Adventure|Animation|Comedy,2016.0,0.701200
4781,4886,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,2001.0,0.700583


In [266]:
print(collab.columns)
print(content.columns)
print(popular.columns)

Index(['movieId', 'title', 'genres', 'year', 'clean_title', 'genres_list',
       'imdbId', 'tmdbId', 'tag', 'combined_features', 'svd_score'],
      dtype='object')
Index(['movieId', 'clean_title', 'genres', 'year', 'similarity_score'], dtype='object')
Index(['movieId', 'clean_title', 'genres', 'average_rating', 'rating_count',
       'weighted_rating'],
      dtype='object')


In [267]:
print(collab.head())
print(content.head())
print(popular.head())

      movieId                                              title  \
Rank                                                               
0         858                              Godfather, The (1972)   
1        1222                           Full Metal Jacket (1987)   
2         924                       2001: A Space Odyssey (1968)   
3         750  Dr. Strangelove or: How I Learned to Stop Worr...   
4        2858                             American Beauty (1999)   

                      genres    year  \
Rank                                   
0                Crime|Drama  1972.0   
1                  Drama|War  1987.0   
2     Adventure|Drama|Sci-Fi  1968.0   
3                 Comedy|War  1964.0   
4              Drama|Romance  1999.0   

                                            clean_title  \
Rank                                                      
0                                        Godfather, The   
1                                     Full Metal Jacket   
2     

In [268]:
collab = (
    collab[
        [
            "movieId",
            "svd_score"
        ]
    ]
    .rename(
        columns={
            "svd_score":"collab_score"
        }
    )
)

In [269]:
content = (
    content[
        [
            "movieId",
            "similarity_score"
        ]
    ]
    .rename(
        columns={
            "similarity_score":"content_score"
        }
    )
)

In [270]:
popular = (
    popular[
        [
            "movieId",
            "average_rating",
            "rating_count",
            "weighted_rating"
        ]
    ]
    .rename(
        columns={
            "weighted_rating":"popularity_score"
        }
    )
)

# Hybrid

In [271]:
hybrid = (
    collab
    .merge(
        content,
        on="movieId",
        how="outer"
    )
    .merge(
        popular,
        on="movieId",
        how="outer"
    )
)

In [272]:
scaler = MinMaxScaler()

In [273]:
score_columns = [
    "collab_score",
    "content_score",
    "popularity_score"
]

for col in score_columns:

    hybrid[col] = scaler.fit_transform(
        hybrid[[col]].fillna(0)
    )

# Genre Recommedation

In [274]:
def build_genre_profile(
    user_id,
    ratings,
    movie_data
):
    """
    Build a weighted genre preference profile
    for one user.

    Higher rated movies contribute more
    than lower rated movies.
    """

    user_ratings = ratings[
        ratings["userId"] == user_id
    ]

    profile = {}

    user_movies = user_ratings.merge(
        movie_data[
            [
                "movieId",
                "genres"
            ]
        ],
        on="movieId",
        how="left"
    )

    for _, row in user_movies.iterrows():

        if pd.isna(row["genres"]):
            continue

        genres = row["genres"].split("|")

        rating = row["rating"]

        for genre in genres:

            if genre == "(no genres listed)":
                continue

            profile[genre] = (
                profile.get(genre, 0)
                + rating
            )

    if len(profile) == 0:
        return {}

    maximum = max(profile.values())

    for genre in profile:

        profile[genre] /= maximum

    return profile

In [275]:
def calculate_genre_score(hybrid_df, genre_profile):

    genre_scores = []

    for _, row in hybrid_df.iterrows():

        if pd.isna(row["genres"]):
            genre_scores.append(0)
            continue

        movie_genres = row["genres"].split("|")

        score = 0

        count = 0

        for genre in movie_genres:

            if genre == "(no genres listed)":
                continue

            score += genre_profile.get(genre, 0)

            count += 1

        if count == 0:
            genre_scores.append(0)

        else:
            genre_scores.append(score / count)

    hybrid_df["genre_score"] = genre_scores

    return hybrid_df

In [276]:
def get_dynamic_weights(
    user_id,
    ratings
):

    if is_new_user(
        user_id,
        ratings
    ):

        return {
            "collab_score":0,
            "content_score":0,
            "popularity_score":0.45,
            "genre_score":0,
            "rating_score":0.25,
            "rating_count_score":0.20,
            "recency_score":0.10
        }

    if is_sparse_user(
        user_id,
        ratings
    ):

        return {
            "collab_score":0.10,
            "content_score":0.40,
            "popularity_score":0.15,
            "genre_score":0.25,
            "rating_score":0.05,
            "rating_count_score":0.03,
            "recency_score":0.02
        }

    return {
        "collab_score":0.20,
        "content_score":0.30,
        "popularity_score":0.15,
        "genre_score":0.20,
        "rating_score":0.05,
        "rating_count_score":0.05,
        "recency_score":0.05
    }

In [277]:
hybrid["rating_score"] = scaler.fit_transform(
    hybrid[
        ["average_rating"]
    ].fillna(0)
)

In [278]:
hybrid["rating_count_score"] = scaler.fit_transform(
    hybrid[
        ["rating_count"]
    ].fillna(0)
)

In [279]:
hybrid = hybrid.merge(
    movie_data[
        ["movieId", "year","genres"]
    ],
    on="movieId",
    how="left"
)

In [280]:
watched_movies = ratings.loc[
    ratings["userId"] == USER_ID,
    "movieId"
]

hybrid = hybrid[
    ~hybrid["movieId"].isin(
        watched_movies
    )
]

In [281]:
genre_profile = build_genre_profile(
    USER_ID,
    ratings,
    movie_data
)

hybrid = calculate_genre_score(
    hybrid,
    genre_profile
)

In [282]:
hybrid["genre_score"] = scaler.fit_transform(
    hybrid[
        ["genre_score"]
    ].fillna(0)
)

In [283]:
CURRENT_YEAR = 2026

hybrid["movie_age"] = (
    CURRENT_YEAR - hybrid["year"]
)

In [284]:
new_movie_mask = (
    hybrid["rating_count"].fillna(0) == 0
)

hybrid.loc[
    new_movie_mask,
    "collab_score"
] = np.nan

hybrid.loc[
    new_movie_mask,
    "popularity_score"
] = np.nan

In [285]:
hybrid["recency_score"] = (
    1 / (1 + hybrid["movie_age"])
)

hybrid["recency_score"] = scaler.fit_transform(
    hybrid[["recency_score"]]
)

In [286]:
hybrid.loc[
    new_movie_mask,
    "movie_age"
] *= 1.20

In [287]:
score_columns = [
    "collab_score",
    "content_score",
    "popularity_score",
    "genre_score",
    "rating_score",
    "rating_count_score",
    "movie_age"
]

In [288]:
def rank_movies(
    df,
    user_id,
    ratings
):

    weights = get_dynamic_weights(
        user_id,
        ratings
    )

    final_scores = []

    for _, row in df.iterrows():

        score = 0
        total_weight = 0

        for feature, weight in weights.items():

            value = row[feature]

            if pd.notna(value):

                score += value * weight

                total_weight += weight

        if total_weight > 0:

            score /= total_weight

        final_scores.append(score)

    df["final_score"] = final_scores

    return df

In [289]:
hybrid = rank_movies(
    hybrid,
    USER_ID,
    ratings
)

In [290]:
def validate_recommendations(
    recommendations,
    watched_movies
):

    assert (
        recommendations["movieId"]
        .duplicated()
        .sum() == 0
    ), "Duplicate recommendations"

    assert (
        len(
            set(
                recommendations["movieId"]
            ).intersection(
                watched_movies
            )
        ) == 0
    ), "Already watched movie recommended"

    assert (
        recommendations["final_score"]
        .isna()
        .sum() == 0
    ), "NaN score detected"

    print("All validation checks passed.")

In [291]:
def get_final_recommendations(df, top_n=500):
    df = df.sort_values(
        "final_score",
        ascending=False
    )

    return df.head(top_n)

final_recommendations = get_final_recommendations(hybrid)
final_recommendations

,movieId,collab_score,content_score,average_rating,rating_count,popularity_score,rating_score,rating_count_score,year,genres,genre_score,movie_age,recency_score,final_score
6,858,1.000000,0.000000,4.317030,66440.0,0.979755,0.970811,0.645493,1972.0,Crime|Drama,0.585635,54.0,0.040280,0.546919
10,1193,0.546209,0.000000,4.204229,44592.0,0.953726,0.945444,0.433231,1975.0,Drama,1.000000,51.0,0.049704,0.523720
31,3114,NaN,1.000000,NaN,NaN,NaN,0.000000,0.000000,1999.0,Adventure|Animation|Children|Comedy|Fantasy,0.141436,32.4,0.197802,0.520273
28,2355,NaN,0.903345,NaN,NaN,NaN,0.000000,0.000000,1998.0,Adventure|Animation|Children|Comedy,0.162983,33.6,0.186737,0.481442
49,157296,NaN,0.785717,NaN,NaN,NaN,0.000000,0.000000,2016.0,Adventure|Animation|Comedy,0.210866,12.0,0.693706,0.480882
38,50872,NaN,0.717347,NaN,NaN,NaN,0.000000,0.000000,2007.0,Animation|Children|Drama,0.339779,22.8,0.326154,0.460719
5,750,0.865680,0.000000,4.202680,32830.0,0.952839,0.945096,0.318958,1964.0,Comedy|War,0.335635,62.0,0.019536,0.447368
46,103141,NaN,0.740759,NaN,NaN,NaN,0.000000,0.000000,2013.0,Adventure|Animation|Comedy,0.210866,15.6,0.518681,0.446669
41,78499,NaN,0.797447,NaN,NaN,NaN,0.000000,0.000000,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.117864,19.2,0.405430,0.435505
34,6377,NaN,0.780009,NaN,NaN,NaN,0.000000,0.000000,2003.0,Adventure|Animation|Children|Comedy,0.162983,27.6,0.251282,0.429482


# Testing

In [292]:
ratings["userId"].value_counts().head(20)

,count
userId,
175325,33332
17035,9577
55653,9178
123465,9044
171795,9016
10202,7748
198515,7594
49305,7488
22744,7372


In [293]:
final_recommendations.head(175325)

,movieId,collab_score,content_score,average_rating,rating_count,popularity_score,rating_score,rating_count_score,year,genres,genre_score,movie_age,recency_score,final_score
6,858,1.000000,0.000000,4.317030,66440.0,0.979755,0.970811,0.645493,1972.0,Crime|Drama,0.585635,54.0,0.040280,0.546919
10,1193,0.546209,0.000000,4.204229,44592.0,0.953726,0.945444,0.433231,1975.0,Drama,1.000000,51.0,0.049704,0.523720
31,3114,NaN,1.000000,NaN,NaN,NaN,0.000000,0.000000,1999.0,Adventure|Animation|Children|Comedy|Fantasy,0.141436,32.4,0.197802,0.520273
28,2355,NaN,0.903345,NaN,NaN,NaN,0.000000,0.000000,1998.0,Adventure|Animation|Children|Comedy,0.162983,33.6,0.186737,0.481442
49,157296,NaN,0.785717,NaN,NaN,NaN,0.000000,0.000000,2016.0,Adventure|Animation|Comedy,0.210866,12.0,0.693706,0.480882
38,50872,NaN,0.717347,NaN,NaN,NaN,0.000000,0.000000,2007.0,Animation|Children|Drama,0.339779,22.8,0.326154,0.460719
5,750,0.865680,0.000000,4.202680,32830.0,0.952839,0.945096,0.318958,1964.0,Comedy|War,0.335635,62.0,0.019536,0.447368
46,103141,NaN,0.740759,NaN,NaN,NaN,0.000000,0.000000,2013.0,Adventure|Animation|Comedy,0.210866,15.6,0.518681,0.446669
41,78499,NaN,0.797447,NaN,NaN,NaN,0.000000,0.000000,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.117864,19.2,0.405430,0.435505
34,6377,NaN,0.780009,NaN,NaN,NaN,0.000000,0.000000,2003.0,Adventure|Animation|Children|Comedy,0.162983,27.6,0.251282,0.429482


In [294]:
watched_movies = ratings.loc[
    ratings["userId"] ==175325,
    "movieId"
]

print(watched_movies.tolist())

[1, 2, 5, 6, 7, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 55, 57, 58, 59, 60, 62, 63, 65, 68, 69, 70, 71, 72, 73, 76, 77, 78, 79, 80, 81, 82, 85, 86, 89, 90, 92, 94, 95, 96, 97, 99, 100, 101, 104, 105, 106, 110, 111, 112, 114, 115, 116, 117, 119, 121, 122, 123, 124, 125, 127, 128, 130, 132, 133, 134, 136, 138, 141, 143, 144, 145, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 161, 162, 163, 164, 165, 166, 167, 168, 170, 171, 172, 173, 175, 176, 177, 178, 180, 181, 183, 184, 185, 187, 188, 190, 193, 194, 195, 196, 198, 199, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 213, 214, 215, 216, 218, 219, 220, 223, 224, 225, 227, 229, 230, 231, 232, 233, 234, 235, 236, 237, 239, 242, 245, 246, 247, 249, 252, 253, 254, 256, 259, 260, 261, 262, 263, 264, 265, 266, 268, 269, 271, 272, 273, 279, 280, 281, 282, 283, 286, 288, 289, 290, 292, 293, 294, 296, 297, 298, 

In [295]:
set(final_recommendations["movieId"]).intersection(set(watched_movies))

{50,
 58,
 296,
 318,
 750,
 858,
 904,
 924,
 1193,
 1207,
 1222,
 1230,
 1250,
 1252,
 1265,
 1293,
 1299,
 1307,
 1394,
 1617,
 1704,
 2019,
 2248,
 2355,
 2858,
 2959,
 3114,
 4886,
 5618,
 6377,
 8961,
 44555,
 45517,
 50872,
 68954,
 72356,
 78499,
 87876,
 95375,
 95856,
 95858,
 103141,
 109420,
 126405,
 157296,
 159817,
 170705,
 171011,
 175831,
 201588,
 202439}

In [296]:
final_recommendations["movieId"].duplicated().sum()

np.int64(0)

In [297]:
final_recommendations.isna().sum()

,0
movieId,0
collab_score,36
content_score,0
average_rating,36
rating_count,36
popularity_score,36
rating_score,0
rating_count_score,0
year,0
genres,0


In [298]:
final_recommendations.head(99999999)

,movieId,collab_score,content_score,average_rating,rating_count,popularity_score,rating_score,rating_count_score,year,genres,genre_score,movie_age,recency_score,final_score
6,858,1.000000,0.000000,4.317030,66440.0,0.979755,0.970811,0.645493,1972.0,Crime|Drama,0.585635,54.0,0.040280,0.546919
10,1193,0.546209,0.000000,4.204229,44592.0,0.953726,0.945444,0.433231,1975.0,Drama,1.000000,51.0,0.049704,0.523720
31,3114,NaN,1.000000,NaN,NaN,NaN,0.000000,0.000000,1999.0,Adventure|Animation|Children|Comedy|Fantasy,0.141436,32.4,0.197802,0.520273
28,2355,NaN,0.903345,NaN,NaN,NaN,0.000000,0.000000,1998.0,Adventure|Animation|Children|Comedy,0.162983,33.6,0.186737,0.481442
49,157296,NaN,0.785717,NaN,NaN,NaN,0.000000,0.000000,2016.0,Adventure|Animation|Comedy,0.210866,12.0,0.693706,0.480882
38,50872,NaN,0.717347,NaN,NaN,NaN,0.000000,0.000000,2007.0,Animation|Children|Drama,0.339779,22.8,0.326154,0.460719
5,750,0.865680,0.000000,4.202680,32830.0,0.952839,0.945096,0.318958,1964.0,Comedy|War,0.335635,62.0,0.019536,0.447368
46,103141,NaN,0.740759,NaN,NaN,NaN,0.000000,0.000000,2013.0,Adventure|Animation|Comedy,0.210866,15.6,0.518681,0.446669
41,78499,NaN,0.797447,NaN,NaN,NaN,0.000000,0.000000,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.117864,19.2,0.405430,0.435505
34,6377,NaN,0.780009,NaN,NaN,NaN,0.000000,0.000000,2003.0,Adventure|Animation|Children|Comedy,0.162983,27.6,0.251282,0.429482


In [299]:
ratings["userId"].value_counts().sort_values().head(20)

,count
userId,
183776,20
192976,20
11855,20
103561,20
119491,20
103572,20
4618,20
169930,20
192998,20


In [300]:
final_recommendations.head(119546)

,movieId,collab_score,content_score,average_rating,rating_count,popularity_score,rating_score,rating_count_score,year,genres,genre_score,movie_age,recency_score,final_score
6,858,1.000000,0.000000,4.317030,66440.0,0.979755,0.970811,0.645493,1972.0,Crime|Drama,0.585635,54.0,0.040280,0.546919
10,1193,0.546209,0.000000,4.204229,44592.0,0.953726,0.945444,0.433231,1975.0,Drama,1.000000,51.0,0.049704,0.523720
31,3114,NaN,1.000000,NaN,NaN,NaN,0.000000,0.000000,1999.0,Adventure|Animation|Children|Comedy|Fantasy,0.141436,32.4,0.197802,0.520273
28,2355,NaN,0.903345,NaN,NaN,NaN,0.000000,0.000000,1998.0,Adventure|Animation|Children|Comedy,0.162983,33.6,0.186737,0.481442
49,157296,NaN,0.785717,NaN,NaN,NaN,0.000000,0.000000,2016.0,Adventure|Animation|Comedy,0.210866,12.0,0.693706,0.480882
38,50872,NaN,0.717347,NaN,NaN,NaN,0.000000,0.000000,2007.0,Animation|Children|Drama,0.339779,22.8,0.326154,0.460719
5,750,0.865680,0.000000,4.202680,32830.0,0.952839,0.945096,0.318958,1964.0,Comedy|War,0.335635,62.0,0.019536,0.447368
46,103141,NaN,0.740759,NaN,NaN,NaN,0.000000,0.000000,2013.0,Adventure|Animation|Comedy,0.210866,15.6,0.518681,0.446669
41,78499,NaN,0.797447,NaN,NaN,NaN,0.000000,0.000000,2010.0,Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.117864,19.2,0.405430,0.435505
34,6377,NaN,0.780009,NaN,NaN,NaN,0.000000,0.000000,2003.0,Adventure|Animation|Children|Comedy,0.162983,27.6,0.251282,0.429482
